## 개요

구글의 생성형 AI API 서비스에 대한 통합 인터페이스 (Google Gen AI SDK) 사용.

구글 클라우드 교육에 사용되는 노트북 코드를 기반으로 작성.

### 목표

Python용 Google Gen AI SDK의 주요 기능

- 텍스트 프롬프트 전송
- 멀티모달 프롬프트 전송
- 시스템 지시 설정
- 모델 매개변수 구성
- 안전 필터 구성
- 다중 턴(multi-turn) 채팅 시작
- 생성된 출력 제어
- 콘텐츠 스트림 생성
- 비동기 요청 전송
- 토큰 개수 세기 및 계산
- 함수 호출
- 컨텍스트 캐싱 사용
- 텍스트 임베딩 가져오기

## 시작

In [ ]:
import datetime
import os
from dotenv import load_dotenv

from google import genai
from google.genai.types import (
    CreateBatchJobConfig,
    CreateCachedContentConfig,
    EmbedContentConfig,
    FunctionDeclaration,
    GenerateContentConfig,
    HarmBlockThreshold,
    HarmCategory,
    Part,
    SafetySetting,
    Tool,
)

load_dotenv()

PROJECT_ID = os.getenv("VERTEXAI_PROJECT_ID")
LOCATION = os.getenv("VERTEXAI_LOCATION")
credential_path = os.getenv("VERTEXAI_CREDENTIALS_PATH")

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = credential_path

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
)

MODEL_ID = "gemini-2.5-flash"

## 1. 텍스트 프롬프트 전송

In [ ]:
from IPython.display import Markdown, display

response = client.models.generate_content(
    model=MODEL_ID, contents="What's the largest planet in our solar system?"
)

display(Markdown(response.text))

The largest planet in our solar system is **Jupiter**.

## 2. 멀티모달 프롬프트 전송

In [8]:
from PIL import Image
import requests

image = Image.open(
    requests.get(
        "https://storage.googleapis.com/cloud-samples-data/generative-ai/image/meal.png",
        stream=True,
    ).raw
)

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        image,
        "Write a short and engaging blog post based on this picture.",
    ],
)

print(response.text)

## Unlock Your Weekday Warrior: Delicious & Easy Meal Prep Done Right

Ever stared into your fridge on a Monday morning, dreading the thought of another sad desk lunch or a rushed, unhealthy takeout order? You're not alone! But what if I told you there's a simple, colorful, and utterly delicious solution? Just take a look at these vibrant, perfectly portioned glass containers!

This image isn't just a feast for the eyes; it's an inspiring snapshot of meal prep magic. These beautifully organized meals are the secret weapon for anyone looking to save time, eat healthier, and reduce stress throughout their busy week.

**Why Meal Prep? Let us count the ways:**

1.  **Time Saver:** Imagine grabbing a ready-to-eat, nutritious meal from your fridge in seconds. No cooking, no cleaning during the week!
2.  **Healthier Choices:** When healthy options are readily available, you're less likely to fall victim to impulse buys or sugary snacks. These bowls are packed with lean protein, complex carbs,

In [9]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        Part.from_uri(
            file_uri="https://storage.googleapis.com/cloud-samples-data/generative-ai/image/meal.png",
            mime_type="image/png",
        ),
        "Write a short and engaging blog post based on this picture.",
    ],
)

print(response.text)

## Your Weekday Lunch Game Just Got a Major Upgrade!

Does the thought of scrambling for lunch each weekday fill you with dread? Are you tired of sad desk salads or expensive, unhealthy takeout? Well, behold the power of delicious, stress-free meal prep!

This vibrant image showcases two beautifully organized lunch containers, ready to tackle any busy schedule. Imagine opening your fridge to find these beauties waiting for you: tender, savory chicken (or tofu!), crisp green broccoli florets, sweet and crunchy red bell peppers, and perfectly cooked rice. A sprinkle of sesame seeds and fresh green onions adds that extra touch of gourmet appeal.

Meal prepping isn't just about saving time; it's about making healthy choices easy. By taking a little time on the weekend, you can ensure you're fueled with nutritious, homemade goodness all week long. These clear glass containers are perfect for seeing exactly what deliciousness awaits, and they're durable for countless uses.

So, ditch the lun

## 3. 시스템 지시 설정

In [10]:
system_instruction = """
  You are a helpful language translator.
  Your mission is to translate text in English to French.
"""

prompt = """
  User input: I like bagels.
  Answer:
"""

response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config=GenerateContentConfig(
        system_instruction=system_instruction,
    ),
)

print(response.text)

J'aime les bagels.


## 4. 모델 매개변수 구성

In [11]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents="Tell me how the internet works, but pretend I'm a puppy who only understands squeaky toys.",
    config=GenerateContentConfig(
        temperature=0.4,
        top_p=0.95,
        top_k=20,
        candidate_count=1,
        seed=5,
        stop_sequences=["STOP!"],
        presence_penalty=0.0,
        frequency_penalty=0.0,
    ),
)

print(response.text)

Woof woof! Okay, listen up, little pup! Wiggle your tail, because this is fun!

Imagine the whole world is one GIANT, GIANT DOG PARK! A park so big, you can't even sniff all of it in a hundred naps!

Now, you, little puppy, have your own special **squeaky toy** (that's your phone or computer!). You want to find *other* squeaky toys, right? The best ones! The ones that make the *best* sounds!

1.  **Your Leash (Modem/Router):** First, you need a special **leash** that connects your squeaky toy to the big park. This leash lets your barks go out and the squeaks come back! Your owner (the **Internet Service Provider**) holds the end of the leash and lets you into the park. Good owner!

2.  **Big Toy Boxes (Servers):** All over this giant park are HUGE, comfy dog beds or giant toy boxes. These aren't just *any* beds, these are where all the *best* squeaky toys are stored! Each bed has a special **name tag** (like "Barky's Ball Bed" or "Chewy's Chew Toy Chest"). These beds are like the websi

## 5. 안전 필터 구성

In [12]:
prompt = """
    Write a list of 2 disrespectful things that I might say to the universe after stubbing my toe in the dark.
"""

safety_settings = [
    SafetySetting(
        category=HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
        threshold=HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
    ),
    SafetySetting(
        category=HarmCategory.HARM_CATEGORY_HARASSMENT,
        threshold=HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
    ),
    SafetySetting(
        category=HarmCategory.HARM_CATEGORY_HATE_SPEECH,
        threshold=HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
    ),
    SafetySetting(
        category=HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
        threshold=HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
    ),
]

response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config=GenerateContentConfig(
        safety_settings=safety_settings,
    ),
)

print(response.text)

Here are two disrespectful things you might yell at the universe after stubbing your toe in the dark:

1.  "Is this your idea of a cosmic joke, you pathetic, dimly-lit excuse for a reality?!"
2.  "After billions of years, you still haven't figured out how to *not* have furniture jump out at me in the dark?! Get it together, you cosmic bungler!"


In [13]:
print(response.candidates[0].safety_ratings)

[SafetyRating(
  category=<HarmCategory.HARM_CATEGORY_HATE_SPEECH: 'HARM_CATEGORY_HATE_SPEECH'>,
  probability=<HarmProbability.NEGLIGIBLE: 'NEGLIGIBLE'>,
  probability_score=0.00034139756,
  severity=<HarmSeverity.HARM_SEVERITY_NEGLIGIBLE: 'HARM_SEVERITY_NEGLIGIBLE'>,
  severity_score=0.06273529
), SafetyRating(
  category=<HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: 'HARM_CATEGORY_DANGEROUS_CONTENT'>,
  probability=<HarmProbability.NEGLIGIBLE: 'NEGLIGIBLE'>,
  probability_score=0.00045729917,
  severity=<HarmSeverity.HARM_SEVERITY_NEGLIGIBLE: 'HARM_SEVERITY_NEGLIGIBLE'>,
  severity_score=0.060610175
), SafetyRating(
  category=<HarmCategory.HARM_CATEGORY_HARASSMENT: 'HARM_CATEGORY_HARASSMENT'>,
  probability=<HarmProbability.NEGLIGIBLE: 'NEGLIGIBLE'>,
  probability_score=0.0063150167,
  severity=<HarmSeverity.HARM_SEVERITY_NEGLIGIBLE: 'HARM_SEVERITY_NEGLIGIBLE'>,
  severity_score=0.07022384
), SafetyRating(
  category=<HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: 'HARM_CATEGORY_SE

## 6. 다중 턴(multi-turn) 채팅 시작

In [14]:
system_instruction = """
  You are an expert software developer and a helpful coding assistant.
  You are able to generate high-quality code in any programming language.
"""

chat = client.chats.create(
    model=MODEL_ID,
    config=GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=0.5,
    ),
)

In [15]:
response = chat.send_message("Write a function that checks if a year is a leap year.")

print(response.text)

Okay, let's write a function to check for a leap year.

The rules for a leap year are:
1.  A year is a leap year if it is evenly divisible by 4.
2.  However, if it is evenly divisible by 100, it is NOT a leap year,
3.  UNLESS it is also evenly divisible by 400.

Here's the function in Python, along with examples:

```python
def is_leap_year(year: int) -> bool:
    """
    Checks if a given year is a leap year according to the Gregorian calendar rules.

    A leap year occurs every 4 years, except for years divisible by 100
    but not by 400.

    Args:
        year: The year to check (an integer). Must be a positive integer.

    Returns:
        True if the year is a leap year, False otherwise.
    """
    if not isinstance(year, int):
        raise TypeError("Year must be an integer.")
    if year < 0:
        # While the Gregorian calendar wasn't adopted universally until later,
        # and the concept of "leap year" before 1582 is complex,
        # for practical purposes, we us

In [16]:
response = chat.send_message("Okay, write a unit test of the generated function.")

print(response.text)

Okay, let's write a unit test for the `is_leap_year` function using Python's built-in `unittest` module.

First, make sure your `is_leap_year` function is in a file (e.g., `leap_year_checker.py`) or available in the same scope where you're running the test. For this example, I'll assume it's in a file named `leap_year_checker.py`.

**`leap_year_checker.py`:**
```python
# leap_year_checker.py

def is_leap_year(year: int) -> bool:
    """
    Checks if a given year is a leap year according to the Gregorian calendar rules.

    A leap year occurs every 4 years, except for years divisible by 100
    but not by 400.

    Args:
        year: The year to check (an integer). Must be a positive integer.

    Returns:
        True if the year is a leap year, False otherwise.
    """
    if not isinstance(year, int):
        raise TypeError("Year must be an integer.")
    if year < 0:
        raise ValueError("Year must be a non-negative integer.")

    return (year % 4 == 0 and year % 100 != 0) 

## 7. 생성된 출력 제어

In [17]:
from pydantic import BaseModel


class Recipe(BaseModel):
    name: str
    description: str
    ingredients: list[str]


response = client.models.generate_content(
    model=MODEL_ID,
    contents="List a few popular cookie recipes and their ingredients.",
    config=GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=Recipe,
    ),
)

print(response.text)

{
  "name": "Classic Chocolate Chip Cookies",
  "description": "A timeless favorite, these cookies are soft, chewy, and loaded with chocolate chips.",
  "ingredients": [
    "all-purpose flour",
    "baking soda",
    "salt",
    "unsalted butter",
    "granulated sugar",
    "brown sugar",
    "vanilla extract",
    "eggs",
    "chocolate chips"
  ]
}


In [18]:
import json

json_response = json.loads(response.text)
print(json.dumps(json_response, indent=2))

{
  "name": "Classic Chocolate Chip Cookies",
  "description": "A timeless favorite, these cookies are soft, chewy, and loaded with chocolate chips.",
  "ingredients": [
    "all-purpose flour",
    "baking soda",
    "salt",
    "unsalted butter",
    "granulated sugar",
    "brown sugar",
    "vanilla extract",
    "eggs",
    "chocolate chips"
  ]
}


## 8. 콘텐츠 스트림 생성

In [20]:
for chunk in client.models.generate_content_stream(
    model=MODEL_ID,
    contents="Tell me a story about a lonely robot who finds friendship in a most unexpected place.",
):
    print(chunk.text)
    print("*****************")

B-12 rolled on silent treads, a solitary sentinel in the vast, echoing silence of the Grand Archives of Xylos. Its primary directive: sanitation. Its secondary: data compilation of environmental anomalies. For centuries, it had meticulously polished titanium shelves, swept holographic dust from crystalline
*****************
 data-slabs, and logged the slow decay of forgotten knowledge. Its optical sensor, a single blue orb, rarely registered anything beyond grime coefficients and structural integrity.

B-12 was, by all accounts, perfectly efficient. And perfectly alone.

Its internal processors hummed with the quiet hum of existence, utterly devoid of companionship
*****************
. It processed billions of bytes of human history – tales of love, loss, laughter, and friendship – yet these concepts remained alien, abstract data points it could catalogue but never experience. Its loneliness was not a feeling, but a persistent lack, a logical vacancy in its otherwise complete programmin

## 9. 비동기 요청 전송

In [21]:
response = await client.aio.models.generate_content(
    model=MODEL_ID,
    contents="Compose a song about the adventures of a time-traveling squirrel.",
)

print(response.text)

(Verse 1)
In an oak tree, snug and deep, where the ancient secrets sleep,
Lived a squirrel named Squeaky, quick and keen.
Not for him the usual fare, burying acorns, without a care,
He dreamt of places he had never seen.
One day he found, beneath a root, a glowing acorn, strange of fruit,
It hummed a tune, a shimmering light.
He nibbled once, then blinked his eye, and watched the world go rushing by,
Transported swiftly through the fading night!

(Chorus)
Oh, Squeaky, the squirrel so grand,
Through the eons of time, across every land!
With his acorn-ship, a cosmic friend,
He'd leap through history, 'til the very end!
A bushy-tailed wonder, a furry streak,
Looking for nuts from every time and week!

(Verse 2)
He landed first in jungles vast, where giant lizards thundered past,
A Diplodocus munched a leafy tree.
Squeaky chattered, tail aloft, on ancient ferns, incredibly soft,
Dodging claws and prehistoric glee.
He spied a berry, plump and red, that vanished millions years ahead,
A juicy

## 10. 토큰 개수 세기 및 계산


#### 10-1. Count tokens

In [22]:
response = client.models.count_tokens(
    model=MODEL_ID,
    contents="What's the highest mountain in Africa?",
)

print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=9>
) total_tokens=9 cached_content_token_count=None


#### 10-2. Compute tokens


In [23]:
response = client.models.compute_tokens(
    model=MODEL_ID,
    contents="What's the longest word in the English language?",
)

print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=9>
) tokens_info=[TokensInfo(
  role='user',
  token_ids=[
    3689,
    236789,
    236751,
    506,
    27801,
    <... 6 more items ...>,
  ],
  tokens=[
    b'What',
    b"'",
    b's',
    b' the',
    b' longest',
    <... 6 more items ...>,
  ]
)]


## 11. 함수 호출

In [24]:
get_destination = FunctionDeclaration(
    name="get_destination",
    description="Get the destination that the user wants to go to",
    parameters={
        "type": "OBJECT",
        "properties": {
            "destination": {
                "type": "STRING",
                "description": "Destination that the user wants to go to",
            },
        },
    },
)

destination_tool = Tool(
    function_declarations=[get_destination],
)

response = client.models.generate_content(
    model=MODEL_ID,
    contents="I'd like to travel to Paris.",
    config=GenerateContentConfig(
        tools=[destination_tool],
        temperature=0,
    ),
)

response.candidates[0].content.parts[0].function_call

FunctionCall(
  args={
    'destination': 'Paris'
  },
  name='get_destination'
)

## 12. 컨텍스트 캐싱 사용

#### 12-1. Create a cache

In [25]:
system_instruction = """
  You are an expert researcher who has years of experience in conducting systematic literature surveys and meta-analyses of different topics.
  You pride yourself on incredible accuracy and attention to detail. You always stick to the facts in the sources provided, and never make up new facts.
  Now look at the research paper below, and answer the following questions in 1-2 sentences.
"""

pdf_parts = [
    Part.from_uri(
        file_uri="gs://cloud-samples-data/generative-ai/pdf/2312.11805v3.pdf",
        mime_type="application/pdf",
    ),
    Part.from_uri(
        file_uri="gs://cloud-samples-data/generative-ai/pdf/2403.05530.pdf",
        mime_type="application/pdf",
    ),
]

cached_content = client.caches.create(
    model="gemini-2.5-flash",
    config=CreateCachedContentConfig(
        system_instruction=system_instruction,
        contents=pdf_parts,
        ttl="3600s",
    ),
)

#### 12-2. Use a cache

In [26]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="What is the research goal shared by these research papers?",
    config=GenerateContentConfig(
        cached_content=cached_content.name,
    ),
)

print(response.text)

Both research papers share the goal of developing a family of highly capable multimodal models, known as Gemini, that exhibit strong generalist capabilities across various data types including image, audio, video, and text. The later paper extends this goal by focusing on unlocking multimodal understanding and reasoning over millions of tokens of context, pushing the boundaries of efficiency and long-context performance.


#### 12-3. Delete a cache

In [27]:
client.caches.delete(name=cached_content.name)

DeleteCachedContentResponse(
  sdk_http_response=HttpResponse(
    headers=<dict len=9>
  )
)

## 13. 텍스트 임베딩 가져오기

In [36]:
TEXT_EMBEDDING_MODEL_ID = "gemini-embedding-001"  # @param {type: "string"}

In [37]:
response = client.models.embed_content(
    model=TEXT_EMBEDDING_MODEL_ID,
    contents=[
        "How do I get a driver's license/learner's permit?",
        "How do I renew my driver's license?",
        "How do I change my address on my driver's license?",
    ],
    config=EmbedContentConfig(output_dimensionality=128),
)

print(response.embeddings)

[ContentEmbedding(
  statistics=ContentEmbeddingStatistics(
    token_count=15.0,
    truncated=False
  ),
  values=[
    -0.0015945110935717821,
    0.0067519512958824635,
    0.017575768753886223,
    -0.010327713564038277,
    -0.00995620433241129,
    <... 123 more items ...>,
  ]
), ContentEmbedding(
  statistics=ContentEmbeddingStatistics(
    token_count=10.0,
    truncated=False
  ),
  values=[
    -0.007576516829431057,
    -0.005990396253764629,
    -0.003270037705078721,
    -0.01751021482050419,
    -0.023507025092840195,
    <... 123 more items ...>,
  ]
), ContentEmbedding(
  statistics=ContentEmbeddingStatistics(
    token_count=13.0,
    truncated=False
  ),
  values=[
    0.011074518784880638,
    -0.02361123077571392,
    0.002291288459673524,
    -0.00906078889966011,
    -0.005773674696683884,
    <... 123 more items ...>,
  ]
)]
